In [1]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

In [2]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
from datetime import timedelta, date
#warnings.filterwarnings('ignore')

In [3]:
a = pd.read_excel('./retained_users_20190527.xlsx')
a.columns = ['month','na','ra','ta','nan','ni','ri','ti']
a.head()

,month,na,ra,ta,nan,ni,ri,ti
0,19-07,320833,0.00,320833.00,NaN,45833,0.00,45833.00
1,19-08,320833,32083.30,352916.30,NaN,45833,5499.96,51332.96
2,19-09,320833,57749.94,378582.94,NaN,45833,10083.26,55916.26
3,19-10,495833,73791.59,569624.59,NaN,70833,13749.90,84582.90
4,19-11,495833,109083.24,604916.24,NaN,70833,20208.21,91041.21


In [4]:
a = a.drop(['ta','nan','ti'],axis=1)
a.set_index('month',inplace = True)
a.head()

,na,ra,ni,ri
month,,,,
19-07,320833,0.00,45833,0.00
19-08,320833,32083.30,45833,5499.96
19-09,320833,57749.94,45833,10083.26
19-10,495833,73791.59,70833,13749.90
19-11,495833,109083.24,70833,20208.21


In [5]:
b = pd.read_excel('./Conversion rate (1).xlsx')

In [6]:
b.head()

,(Jul'19-Sep'19),(Oct'19-Dec'19),(Jan'20-Mar'20),(Apr'20-Jun'20),(Jul'20-Sep'20),(Oct'20-Dec'20),(Jan'21-Mar'21),(Apr'21-Jun'21),(Jul'21-Sep'21),(Oct'21-Dec'21)
Avg. Rev per paying user per month - Android,560.0,840.0,910.0,980.0,1050.0,1120.0,1260.0,1260.0,1260.0,1260.0
Avg. Rev per paying user per month - iOS,840.0,1260.0,1365.0,1470.0,1575.0,1680.0,1890.0,1890.0,1890.0,1890.0
NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Conversion Ratio,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df = b.transpose()
df.columns = ['ARPPU_A','ARPPU_I','nan1','nan2','nan3','nan4','new_a','retained_a','na5','new_i','retained_i']

In [8]:
df.head()

,ARPPU_A,ARPPU_I,nan1,nan2,nan3,nan4,new_a,retained_a,na5,new_i,retained_i
(Jul'19-Sep'19),560.0,840.0,NaN,NaN,NaN,NaN,0.003,0.06,NaN,0.006,0.09
(Oct'19-Dec'19),840.0,1260.0,NaN,NaN,NaN,NaN,0.008,0.07,NaN,0.012,0.11
(Jan'20-Mar'20),910.0,1365.0,NaN,NaN,NaN,NaN,0.008,0.07,NaN,0.012,0.11
(Apr'20-Jun'20),980.0,1470.0,NaN,NaN,NaN,NaN,0.009,0.08,NaN,0.013,0.11
(Jul'20-Sep'20),1050.0,1575.0,NaN,NaN,NaN,NaN,0.009,0.08,NaN,0.013,0.11


In [9]:
df_conversion = df[['new_a','retained_a','new_i','retained_i']]
df_conversion.index.name = 'date'

In [10]:
df_conversion.head()

,new_a,retained_a,new_i,retained_i
date,,,,
(Jul'19-Sep'19),0.003,0.06,0.006,0.09
(Oct'19-Dec'19),0.008,0.07,0.012,0.11
(Jan'20-Mar'20),0.008,0.07,0.012,0.11
(Apr'20-Jun'20),0.009,0.08,0.013,0.11
(Jul'20-Sep'20),0.009,0.08,0.013,0.11


In [11]:
split1 = df_conversion['new_a'].tolist()
split2 = df_conversion['retained_a'].tolist()
split3 = df_conversion['new_i'].tolist()
split4 = df_conversion['retained_i'].tolist()

split1 = [item for item in split1 for i in range(3)]
split2 = [item for item in split2 for i in range(3)]
split3 = [item for item in split3 for i in range(3)]
split4 = [item for item in split4 for i in range(3)]

con_month = pd.DataFrame({'new_a':split1,'retained_a':split2,'new_i':split3,'retained_i':split4})

In [15]:
def daterange(date1, date2):
    for n in range(int ((date2 - date1).days)+1):
        yield date1 + timedelta(n)
date_list = []
start_dt = date(19,7,1)
end_dt = date(21,12,31)
for dt in daterange(start_dt, end_dt):
    date_list.append(dt.strftime("%Y-%m"))
    
xy = pd.DataFrame({'months': date_list})
xy.drop_duplicates('months',inplace = True)
xy.reset_index(drop=True, inplace=True)

In [19]:
z = pd.merge(xy,con_month,left_index=True,right_index = True)
z.set_index('months',inplace=True)

In [20]:
paying_users = pd.DataFrame(a.values*z.values, columns=a.columns, index=a.index)

In [22]:
paying_users

,na,ra,ni,ri
month,,,,
19-07,962.499,0.00000,274.998,0.00000
19-08,962.499,1924.99800,274.998,494.99640
19-09,962.499,3464.99640,274.998,907.49340
19-10,3966.664,5165.41130,849.996,1512.48900
19-11,3966.664,7635.82680,849.996,2222.90310
19-12,3966.664,9636.65920,849.996,2827.90090
20-01,4900.000,11157.70005,1050.000,3327.48240
20-02,4900.000,13454.57680,1050.000,3959.98570
20-03,4900.000,15465.61975,1050.000,4477.90530


In [23]:
paying_users['android'] = paying_users['na'] + paying_users['ra']
paying_users['iOS'] = paying_users['ni'] + paying_users['ri']

In [24]:
paying_users.head()

,na,ra,ni,ri,android,iOS
month,,,,,,
19-07,962.499,0.0000,274.998,0.0000,962.4990,274.9980
19-08,962.499,1924.9980,274.998,494.9964,2887.4970,769.9944
19-09,962.499,3464.9964,274.998,907.4934,4427.4954,1182.4914
19-10,3966.664,5165.4113,849.996,1512.4890,9132.0753,2362.4850
19-11,3966.664,7635.8268,849.996,2222.9031,11602.4908,3072.8991


In [25]:
df_arppu = df[['ARPPU_A','ARPPU_I']]

In [26]:
df_arppu

,ARPPU_A,ARPPU_I
date,,
(Jul'19-Sep'19),560.0,840.0
(Oct'19-Dec'19),840.0,1260.0
(Jan'20-Mar'20),910.0,1365.0
(Apr'20-Jun'20),980.0,1470.0
(Jul'20-Sep'20),1050.0,1575.0
(Oct'20-Dec'20),1120.0,1680.0
(Jan'21-Mar'21),1260.0,1890.0
(Apr'21-Jun'21),1260.0,1890.0
(Jul'21-Sep'21),1260.0,1890.0


In [27]:
split_a = df_arppu['ARPPU_A'].tolist()
split_b = df_arppu['ARPPU_I'].tolist()

split_a = [item for item in split_a for i in range(3)]
split_b = [item for item in split_b for i in range(3)]

paid_month = pd.DataFrame({'arppu_a':split_a,'arppu_i':split_b})

In [28]:
paid_month

,arppu_a,arppu_i
0,560.0,840.0
1,560.0,840.0
2,560.0,840.0
3,840.0,1260.0
4,840.0,1260.0
5,840.0,1260.0
6,910.0,1365.0
7,910.0,1365.0
8,910.0,1365.0
9,980.0,1470.0


In [29]:
z1 = pd.merge(xy,paid_month,left_index=True,right_index = True)
z1.set_index('months',inplace=True)

In [30]:
z1.head()

,arppu_a,arppu_i
months,,
19-07,560.0,840.0
19-08,560.0,840.0
19-09,560.0,840.0
19-10,840.0,1260.0
19-11,840.0,1260.0


In [32]:
paying_total = paying_users[['android','iOS']]

In [33]:
paying_total.head()

,android,iOS
month,,
19-07,962.4990,274.9980
19-08,2887.4970,769.9944
19-09,4427.4954,1182.4914
19-10,9132.0753,2362.4850
19-11,11602.4908,3072.8991


In [34]:
paying_users.head()

,na,ra,ni,ri,android,iOS
month,,,,,,
19-07,962.499,0.0000,274.998,0.0000,962.4990,274.9980
19-08,962.499,1924.9980,274.998,494.9964,2887.4970,769.9944
19-09,962.499,3464.9964,274.998,907.4934,4427.4954,1182.4914
19-10,3966.664,5165.4113,849.996,1512.4890,9132.0753,2362.4850
19-11,3966.664,7635.8268,849.996,2222.9031,11602.4908,3072.8991


In [48]:
df3 = pd.DataFrame(paying_total.values*z1.values, columns=z1.columns, index=z1.index)

In [49]:
df3.head()
df3 = df3.astype(int)

In [50]:
df3

,arppu_a,arppu_i
months,,
19-07,538999,230998
19-08,1616998,646795
19-09,2479397,993292
19-10,7670943,2976731
19-11,9746092,3871852
19-12,11426791,4634150
20-01,14612507,5975263
20-02,16702664,6838630
20-03,18532713,7545590


In [51]:
df3['total_revenue'] = df3.sum(axis=1)

In [52]:
df3

,arppu_a,arppu_i,total_revenue
months,,,
19-07,538999,230998,769997
19-08,1616998,646795,2263793
19-09,2479397,993292,3472689
19-10,7670943,2976731,10647674
19-11,9746092,3871852,13617944
19-12,11426791,4634150,16060941
20-01,14612507,5975263,20587770
20-02,16702664,6838630,23541294
20-03,18532713,7545590,26078303


In [55]:
df4 = df3/70

In [57]:
df4 = df4.astype(int)

In [59]:
df4

,arppu_a,arppu_i,total_revenue
months,,,
19-07,7699,3299,10999
19-08,23099,9239,32339
19-09,35419,14189,49609
19-10,109584,42524,152109
19-11,139229,55312,194542
19-12,163239,66202,229442
20-01,208750,85360,294111
20-02,238609,97694,336304
20-03,264753,107794,372547
